# Model Evaluation 

I evaluated the renewal-risk model using both ranking and decision metrics appropriate for imbalanced churn problems. In addition to ROC-AUC and PR-AUC, I assessed threshold-dependent precision/recall tradeoffs and top-K capture rates to reflect Customer Success capacity constraints. I then translated model scores into High / Medium / Low risk buckets to support operational prioritization.

- **ROC-AUC**: How well does the model separate churners from renewers overall? 
- **PR-AUC**: How well does the model identify churners when the positive class is relatively rare? 
- **Precision/Recall**: What tradeoff are we making at a specific threshold? 
- **Recall@K**: If we only intervene on the highest-risk accounts, how many churners do we catch?
- **Risk buckets**: Can we segment the portfolio into action-oriented tiers? 

In [0]:
from pyspark.sql import functions as F 
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array


In [0]:
predictions_scored = spark.table("renewal_model_predictions")
display(
    predictions_scored.select(
        "client_id"
        ,"renewal_cycle_id"
        ,"non_renewal_flag"
        ,"prediction"
        ,"non_renewal_probability"
    )
)

In [0]:
# What for? What I'm looking for? 
display(
    predictions_scored
    .groupBy("non_renewal_flag")
    .count()
)

In [0]:
display(
    predictions_scored
    .agg(
        F.avg("non_renewal_flag").alias("churn_prevalence")
    )
)

In [0]:
# Recompute ROC-AUC and PR-AUC
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="non_renewal_flag",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="non_renewal_flag",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

roc_auc = roc_evaluator.evaluate(predictions_scored)
pr_auc = pr_evaluator.evaluate(predictions_scored)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

In [0]:
# Confusion matrix at default threshold 
# Your prediction column uses the default threshold from logistic regression ??
display(
    predictions_scored
    .groupBy("non_renewal_flag", "prediction")
    .count()
    .orderBy("non_renewal_flag", "prediction")
)

In [0]:
# Compute precision, recall and F1 at default threshold 
cm = (
    predictions_scored
    .groupBy("non_renewal_flag", "prediction")
    .count()
    .collect() # Tp bring all of those rowas back to Python as a local list. Then I can loop through them
)

tp = fp = tn = fn = 0

for row in cm:
    actual = row["non_renewal_flag"]
    pred = row["prediction"]
    cnt = row["count"]
    
    if actual == 1 and pred == 1:
        tp = cnt
    elif actual == 0 and pred == 1:
        fp = cnt
    elif actual == 0 and pred == 0:
        tn = cnt
    elif actual == 1 and pred == 0:
        fn = cnt

precision = tp / (tp + fp) if (tp + fp) > 0 else 0 # It tells you how clean your flagged list is
recall = tp / (tp + fn) if (tp + fn) > 0 else 0 # It tells you how many churners you catch
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("TP:", tp)
print("FP:", fp)
print("TN:", tn)
print("FN:", fn)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

In [0]:
# Evaluate at multiple thresholds 
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]
results = []

for t in thresholds:
    temp = predictions_scored.withColumn(
        "pred_t",
        F.when(F.col("non_renewal_probability") >= F.lit(t), 1).otherwise(0)
    )
    
    cm_t = temp.groupBy("non_renewal_flag", "pred_t").count().collect()
    
    tp = fp = tn = fn = 0
    for row in cm_t:
        actual = row["non_renewal_flag"]
        pred = row["pred_t"]
        cnt = row["count"]
        
        if actual == 1 and pred == 1:
            tp = cnt
        elif actual == 0 and pred == 1:
            fp = cnt
        elif actual == 0 and pred == 0:
            tn = cnt
        elif actual == 1 and pred == 0:
            fn = cnt
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    results.append((float(t), int(tp), int(fp), int(tn), int(fn), float(precision), float(recall), float(f1)))

threshold_eval_df = spark.createDataFrame(
    results,
    ["threshold", "tp", "fp", "tn", "fn", "precision", "recall", "f1"]
)

display(threshold_eval_df.orderBy("threshold"))

In [0]:
# Rank order performance: Precision@K and Recall@K
# I want to know: If I only act on the top-risk accounts, how many churners am I catching? 
from pyspark.sql.window import Window

w = Window.orderBy(F.desc("non_renewal_probability"))

ranked_preds = (
    predictions_scored
    .withColumn("risk_rank", F.row_number().over(w))
)

total_rows = ranked_preds.count()
total_churners = ranked_preds.filter(F.col("non_renewal_flag") == 1).count()

print("Total rows:", total_rows)
print("Total churners:", total_churners)

In [0]:
# Compute metrics at top 10%, 20%, and 30% 
# This will tell me something like "If you focus on the top 20% highest-risk accounts, you capture 65% of the churners"
k_percents = [0.1, 0.2, 0.3]
k_results = []

for kp in k_percents:
    k = int(total_rows * kp)
    
    top_k = ranked_preds.filter(F.col("risk_rank") <= k)
    
    tp_k = top_k.filter(F.col("non_renewal_flag") == 1).count()
    precision_k = tp_k / k if k > 0 else 0
    recall_k = tp_k / total_churners if total_churners > 0 else 0
    
    k_results.append((kp, k, tp_k, float(precision_k), float(recall_k)))

topk_eval_df = spark.createDataFrame(
    k_results,
    ["top_percent", "k_accounts", "true_churners_captured", "precision_at_k", "recall_at_k"]
)

display(topk_eval_df.orderBy("top_percent"))

In [0]:
# Create risk buckets for storytelling
ranked_preds = ranked_preds.withColumn(
    "risk_percentile"
    ,F.col("risk_rank")/F.lit(total_rows)
)

ranked_preds = ranked_preds.withColumn(
    "risk_bucket", 
    F.when(F.col("risk_percentile") <= 0.2, "High")
    .when(F.col("risk_percentile") <= 0.5, "Medium")
    .otherwise("Low")
)

display(
    ranked_preds.select(
        "client_id"
        ,"renewal_cycle_id"
        ,"non_renewal_probability"
        ,"risk_rank"
        ,"risk_bucket"
        ,"non_renewal_flag"
    )
    .orderBy("risk_rank")
)

In [0]:
# Evaluate bucket quality
display(
    ranked_preds
    .groupBy("risk_bucket")
    .agg(
        F.count("*").alias("accounts")
        ,F.sum(F.col("non_renewal_flag")).alias("actual_churners")
        ,F.avg("non_renewal_flag").alias("churn_rate")
        ,F.avg("non_renewal_probability").alias("avg_predicted_risk"))
    .orderBy("risk_bucket")
)

In [0]:
# Inspect top risk accounts for qualitative sanity-check
display(
    ranked_preds
    .select(
        "client_id",
        "renewal_cycle_id",
        "industry",
        "company_size",
        "non_renewal_probability",
        "risk_bucket",
        "non_renewal_flag"
    )
    .orderBy(F.desc("non_renewal_probability"))
)

In [0]:
# Save threshold evaluation outputs 
spark.sql("DROP TABLE IF EXISTS renewal_model_threshold_eval")
threshold_eval_df.write.mode("overwrite").saveAsTable("renewal_model_threshold_eval")

#Save top-K evaluation outputs 
spark.sql("DROP TABLE IF EXISTS renewal_model_topk_eval")
topk_eval_df.write.mode("overwrite").saveAsTable("renewal_model_topk_eval")

#Save ranked predictions with buckets 
spark.sql("DROP TABLE IF EXISTS renewal_model_ranked_predictions")
ranked_preds.write.mode("overwrite").saveAsTable("renewal_model_ranked_predictions")